# Prepare one current dataset from Supabase

Run after downloading, transforming, and uploading cleaned data. All ratings and
repeated user/item events are retained. Shared global time windows are approximately
80% train, 10% validation, and 10% test; equal timestamps always stay together.
No minimum-interaction or warm-start filtering is applied. No evaluation package is used.


In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / "config" / "ml_config.py").is_file():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from data_loader import load_data, train_test_split, DataLoader
from config.ml_config import DATA_OUTPUT_DIR, REVIEW_TABLE_NAME, ITEM_TABLE_NAME, TRAIN_FILENAME, VALIDATION_FILENAME, TEST_FILENAME

output_dir = DATA_OUTPUT_DIR
reviews = load_data(REVIEW_TABLE_NAME)
items = load_data(ITEM_TABLE_NAME, columns=[
    "parent_asin", "item_title", "description", "features", "main_category", "is_free",
])


Fetching from `Review` : 0rows [00:00, ?rows/s]

Fetching from `Item` : 0rows [00:00, ?rows/s]

In [2]:
train, validation, test = train_test_split(reviews)

# Replace the same current files; no dataset versions or manifests.
output_dir.mkdir(parents=True, exist_ok=True)
for filename, frame in (
    (TRAIN_FILENAME, train), (VALIDATION_FILENAME, validation),
    (TEST_FILENAME, test), ("items.parquet", items),
):
    frame.to_parquet(output_dir / filename, index=False)

# Validate the complete saved dataset, including item coverage.
data = DataLoader(output_dir=output_dir)
print({"train": len(data.train_df), "validation": len(data.validation_df), "test": len(data.test_df)})

print(f"Exported dataset to: {output_dir.resolve()}")


{'train': 195181, 'validation': 24343, 'test': 24485}
Exported dataset to: C:\Users\user\OneDrive\Desktop\Amozon\amozon_backend\data\output
